# Subunit B interacting residues workflow

This notebook was executed using the following enviornment:

1. WSL Installation: https://learn.microsoft.com/en-us/windows/wsl/install
2. VS Code Installation: https://code.visualstudio.com/docs/setup/windows#_install-vs-code-on-windows
3. WSL extension on VS Code: https://code.visualstudio.com/docs/remote/wsl

OpenBabel installation:

1. ```sudo apt update```
2. ```sudo apt install openbabel```

Python environment:

1. ```!python -m venv .enzengdpa```
2. ```!pip install requirements.txt```

We evaluated the binding interactions between amino acids and a ligand using output files from different algorithms: AutoDock Vina, DiffDock, and Boltz-2. These algorithms were executed on the Tamarind Bio Platform.

We installed ```DiffDock-Pocket``` on WSL by following developers' instructions: https://github.com/plainerman/DiffDock-Pocket/

DiffDock-Pocket was used to validate the amino acids interacting with the ligand for each algorithm's output. It was executed separately from this notebook once the relevant amino acids were identified.

In [1]:
import os
os.environ['PYTHONDWRITEBYTECODE'] = '0'

# 0. Import packages

In [2]:
# Openbabel terminal subprocess
import subprocess
# Protein & Ligand Pre-Process
import prolif as plf
from rdkit import Chem
from rdkit.Chem import AllChem
import MDAnalysis as mda
# DataFrame Manipulation
import pandas as pd
# Molecular Visualization
from prolif.plotting.network import LigNetwork
import py3Dmol

/home/mreyes/gogec/2025/enzengDPA/.enzengdpa/lib/python3.12/site-packages/prolif/datafiles.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home/mreyes/gogec/2025/enzengDPA/.enzengdpa/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mreyes/gogec/2025/enzengDPA/.enzengdpa/lib/python3.12/site-packages/MDAnalysis/topology/tables.py:52: DeprecationWarning: Deprecated in version 2.8.0
MDAnalysis.topology.tables has been moved to MDAnalysis.guesser.tables. This import point will be removed in MDAnalysis version 3.0.0
  warnings.warn(wmsg, category=DeprecationWarn

# 1. Import AlphaFold-predicted structure

Add hydrogens to structure. 

Make sure to have installed ```openbabel```.

1. ```sudo apt update```
2. ```sudo apt install openbabel```

In [3]:
protein_raw_file = "tamarind_results/subunit_b_alphafold_model.pdb"
protein_hs_file = "interaction_results/subunit_b_with_h.pdb"

try:
    subprocess.run([
        "obabel", 
        protein_raw_file, 
        "-ipdb", 
        "-O", protein_hs_file, 
        "-opdb", 
        "-p", "7.4"
    ], check=True)
    print("Hydrogens added successfully!")
except subprocess.CalledProcessError as e:
    print(f"An error occurred while running OpenBabel: {e}")
except FileNotFoundError:
    print("OpenBabel is not installed or not in your PATH.")

Hydrogens added successfully!


*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is tamarind_results/subunit_b_alphafold_model.pdb)

1 molecule converted


Load structure with hydrogens into ProLIF

In [4]:
# Load the newly created protonated file
u = mda.Universe(protein_hs_file)

# Convert to ProLIF
dpa_synthase = plf.Molecule.from_mda(u)

print(f"Protein loaded with {dpa_synthase.n_residues} residues and explicit hydrogens.")

Protein loaded with 200 residues and explicit hydrogens.


# 2. Autodock Vina Results Analysis

Import ligand binding pose.

In [5]:
ligand_pose_file = "tamarind_results/subunit_b_ligand_autodock_vina.sdf"

lig_rdkit = Chem.SDMolSupplier(ligand_pose_file, removeHs=False)[0]
htpa_vina = plf.Molecule(lig_rdkit)
print(f"Ligand binding pose loaded.")

Ligand binding pose loaded.


Run fingerprint analysis.

In [6]:
fp = plf.Fingerprint()
fp.run_from_iterable([htpa_vina], dpa_synthase)

# Show results
df = fp.to_dataframe()
df.head()

100%|██████████| 1/1 [00:00<00:00, 73.18it/s]


ligand            UNK0                                                     
protein       MET146.A   THR148.A   ASN150.A            ILE151.A           
interaction VdWContact VdWContact HBAcceptor VdWContact  HBDonor VdWContact
Frame                                                                      
0                 True       True       True       True     True       True

Display 2D diagram.

In [7]:
# Generate the network
net = LigNetwork.from_fingerprint(fp, htpa_vina, kind="frame", frame=0)

# Save the network to a file
html_file = "interaction_results/subunit_b_autodock_vina_interactions.html"
net.save(html_file)

print(f"File saved as {html_file}")

# Display
net.display()

File saved as interaction_results/subunit_b_autodock_vina_interactions.html


Retrieve Interacting Residues as a DataFrame.

In [8]:
interaction_counts_vina = df.T.sum(axis=1).reset_index()
interaction_counts_vina.columns = ['Ligand', 'Residue', 'Interaction', 'Count']

# Define the interactions
target_interactions = "HB|Hydrophobic|VdW"

# Filter the dataframe
filtered_interactions = interaction_counts_vina[
    interaction_counts_vina['Interaction'].str.contains(target_interactions, case=False)
]

# Sort by Residue or Count for better readability
filtered_interactions = filtered_interactions.sort_values(by=['Residue', 'Count'], ascending=[True, False])
filtered_interactions.to_csv("interaction_results/subunit_b_autodock_vina_interactions.csv",index=False)

# Display the result
print("Key Interactions (HBonds, Hydrophobic, VdW):")
display(filtered_interactions)

Key Interactions (HBonds, Hydrophobic, VdW):


,Ligand,Residue,Interaction,Count
2,UNK0,ASN150.A,HBAcceptor,1
3,UNK0,ASN150.A,VdWContact,1
4,UNK0,ILE151.A,HBDonor,1
5,UNK0,ILE151.A,VdWContact,1
0,UNK0,MET146.A,VdWContact,1
1,UNK0,THR148.A,VdWContact,1


Visualize 3D structure.

In [9]:
# Identify residues that had any interaction
interacting_res_list = interaction_counts_vina['Residue'].unique()
res_ids = [int(''.join(filter(str.isdigit, res))) for res in interacting_res_list]

# Open Protein Structure
with open(protein_hs_file, "r") as f:
    pdb_data = f.read()

# Open Ligand Structure
with open(ligand_pose_file, "r") as f:
    sdf_data = f.read()

# Initialize the viewer PDB file
viewer = py3Dmol.view(width=800, height=600)
viewer.addModel(pdb_data, "pdb")
viewer.addModel(sdf_data, "sdf")

# Style the Protein (Green); Cartoon Model
viewer.setStyle({'model': 0}, {'cartoon': {'color': 'green', 'opacity': 0.8}})

# Style the Ligand (Blue); Stick Model
viewer.setStyle({'model': 1}, {'stick': {'colorscheme': 'blueCarbon'}})

# Style Interacting Residues (Red)
viewer.addStyle({'model': 0, 'resi': res_ids}, 
                {'stick': {'color': 'red'}, 'cartoon': {'color': 'red'}})

# Add labels for the interacting residues
for res_name in interacting_res_list:
    res_num = int(''.join(filter(str.isdigit, res_name)))
    viewer.addLabel(res_name, 
                    {'fontSize': 10, 'fontColor': 'white', 'backgroundColor': 'black'},
                    {'model': 0, 'resi': res_num})

viewer.zoomTo({'model': 1})
viewer.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# 3. DiffDock Results Analysis

Import ligand binding pose.

In [10]:
ligand_pose_file = "tamarind_results/subunit_b_ligand_diffdock.sdf"

lig_rdkit = Chem.SDMolSupplier(ligand_pose_file, removeHs=False)[0]
htpa_diffdock = plf.Molecule(lig_rdkit)
print(f"Ligand binding pose loaded.")

Ligand binding pose loaded.


Run fingerprint analysis.

In [11]:
fp = plf.Fingerprint()
fp.run_from_iterable([htpa_diffdock], dpa_synthase)

# Show results
df = fp.to_dataframe()
df.head()

100%|██████████| 1/1 [00:00<00:00, 78.14it/s]


ligand            UNK0                                                         \
protein        THR42.A    PHE43.A    ASN44.A               ASN97.A    SER98.A   
interaction VdWContact VdWContact HBAcceptor VdWContact VdWContact VdWContact   
Frame                                                                           
0                 True       True       True       True       True       True   

ligand                                        
protein       LYS101.A              ASP108.A  
interaction HBAcceptor VdWContact VdWContact  
Frame                                         
0                 True       True       True

Display 2D diagram.

In [12]:
# Generate the network
net = LigNetwork.from_fingerprint(fp, htpa_vina, kind="frame", frame=0)

# Save the network to a file
html_file = "interaction_results/subunit_b_diffdock_interactions.html"
net.save(html_file)

print(f"File saved as {html_file}")

# Display
net.display()

File saved as interaction_results/subunit_b_diffdock_interactions.html


Retrieve Interacting Residues as a DataFrame.

In [13]:
interaction_counts_diffdock = df.T.sum(axis=1).reset_index()
interaction_counts_diffdock.columns = ['Ligand', 'Residue', 'Interaction', 'Count']

# Define the interactions
target_interactions = "HB|Hydrophobic|VdW"

# Filter the dataframe
filtered_interactions = interaction_counts_diffdock[
    interaction_counts_diffdock['Interaction'].str.contains(target_interactions, case=False)
]

# Sort by Residue or Count for better readability
filtered_interactions = filtered_interactions.sort_values(by=['Residue', 'Count'], ascending=[True, False])
filtered_interactions.to_csv("interaction_results/subunit_b_diffdock_interactions.csv",index=False)

# Display the result
print("Key Interactions (HBonds, Hydrophobic, VdW):")
display(filtered_interactions)

Key Interactions (HBonds, Hydrophobic, VdW):


,Ligand,Residue,Interaction,Count
2,UNK0,ASN44.A,HBAcceptor,1
3,UNK0,ASN44.A,VdWContact,1
4,UNK0,ASN97.A,VdWContact,1
8,UNK0,ASP108.A,VdWContact,1
6,UNK0,LYS101.A,HBAcceptor,1
7,UNK0,LYS101.A,VdWContact,1
1,UNK0,PHE43.A,VdWContact,1
5,UNK0,SER98.A,VdWContact,1
0,UNK0,THR42.A,VdWContact,1


Visualize 3D structure.

In [14]:
# Identify residues that had any interaction
interacting_res_list = interaction_counts_diffdock['Residue'].unique()
res_ids = [int(''.join(filter(str.isdigit, res))) for res in interacting_res_list]

# Open Protein Structure
with open(protein_hs_file, "r") as f:
    pdb_data = f.read()

# Open Ligand Structure
with open(ligand_pose_file, "r") as f:
    sdf_data = f.read()

# Initialize the viewer PDB file
viewer = py3Dmol.view(width=800, height=600)
viewer.addModel(pdb_data, "pdb")
viewer.addModel(sdf_data, "sdf")

# Style the Protein (Green); Cartoon Model
viewer.setStyle({'model': 0}, {'cartoon': {'color': 'green', 'opacity': 0.8}})

# Style the Ligand (Blue); Stick Model
viewer.setStyle({'model': 1}, {'stick': {'colorscheme': 'blueCarbon'}})

# Style Interacting Residues (Red)
viewer.addStyle({'model': 0, 'resi': res_ids}, 
                {'stick': {'color': 'red'}, 'cartoon': {'color': 'red'}})

# Add labels for the interacting residues
for res_name in interacting_res_list:
    res_num = int(''.join(filter(str.isdigit, res_name)))
    viewer.addLabel(res_name, 
                    {'fontSize': 10, 'fontColor': 'white', 'backgroundColor': 'black'},
                    {'model': 0, 'resi': res_num})

viewer.zoomTo()
viewer.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# 4. Boltz-2 Results Analysis

In [15]:
docked_pdb = "tamarind_results/subunit_b_complex_boltz.pdb" 

# Load the complex
u = mda.Universe(docked_pdb)

# Separate protein and ligand
ligand_atoms = u.select_atoms("not protein and not resname SOL WAT")

temp_lig_pdb = "interaction_results/aubunit_b_ligand_boltz.pdb"
ligand_atoms.atoms.write(temp_lig_pdb)

lig_rdkit_temp = Chem.MolFromPDBFile(temp_lig_pdb, removeHs=False)

ligand_pose_file = "interaction_results/subunit_b_ligand_boltz.sdf"
with Chem.SDWriter(ligand_pose_file) as writer:
    writer.write(lig_rdkit_temp)
print(f"Ligand successfully converted and stored as SDF.")

Ligand successfully converted and stored as SDF.


/home/mreyes/gogec/2025/enzengDPA/.enzengdpa/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:885: UserWarning: Unit cell dimensions not found. CRYST1 record set to unitary values.
  warnings.warn(
/home/mreyes/gogec/2025/enzengDPA/.enzengdpa/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(


Import ligand binding pose.

In [16]:
lig_rdkit = Chem.SDMolSupplier(ligand_pose_file, removeHs=False)[0]
htpa_boltz = plf.Molecule(lig_rdkit)
print(f"Ligand binding pose loaded.")

Ligand binding pose loaded.


Run fingerprint analysis.

In [17]:
fp = plf.Fingerprint()
fp.run_from_iterable([htpa_boltz], dpa_synthase)

# Show results
df = fp.to_dataframe()
df.head()

100%|██████████| 1/1 [00:00<00:00, 54.02it/s]


ligand            UNK0                                                       
protein        TYR20.A    VAL23.A    PHE24.A    ILE27.A    TRP58.A    ILE62.A
interaction VdWContact VdWContact VdWContact VdWContact VdWContact VdWContact
Frame                                                                        
0                 True       True       True       True       True       True

Display 2D diagram.

In [18]:
# Generate the network
net = LigNetwork.from_fingerprint(fp, htpa_vina, kind="frame", frame=0)

# Save the network to a file
html_file = "interaction_results/subunit_a_boltz_interactions.html"
net.save(html_file)

print(f"File saved as {html_file}")

# Display
net.display()

File saved as interaction_results/subunit_a_boltz_interactions.html


Retrieve Interacting Residues as DataFrame.

In [19]:
interaction_counts_diffdock = df.T.sum(axis=1).reset_index()
interaction_counts_diffdock.columns = ['Ligand', 'Residue', 'Interaction', 'Count']

# Define the interactions
target_interactions = "HB|Hydrophobic|VdW"

# Filter the dataframe
filtered_interactions = interaction_counts_diffdock[
    interaction_counts_diffdock['Interaction'].str.contains(target_interactions, case=False)
]

# Sort by Residue or Count for better readability
filtered_interactions = filtered_interactions.sort_values(by=['Residue', 'Count'], ascending=[True, False])
filtered_interactions.to_csv("interaction_results/subunit_b_boltz_interactions.csv",index=False)

# Display the result
print("Key Interactions (HBonds, Hydrophobic, VdW):")
display(filtered_interactions)

Key Interactions (HBonds, Hydrophobic, VdW):


,Ligand,Residue,Interaction,Count
3,UNK0,ILE27.A,VdWContact,1
5,UNK0,ILE62.A,VdWContact,1
2,UNK0,PHE24.A,VdWContact,1
4,UNK0,TRP58.A,VdWContact,1
0,UNK0,TYR20.A,VdWContact,1
1,UNK0,VAL23.A,VdWContact,1


Visualize 3D structure.

In [20]:
# Identify residues that had any interaction
interacting_res_list = interaction_counts_diffdock['Residue'].unique()
res_ids = [int(''.join(filter(str.isdigit, res))) for res in interacting_res_list]

# Open Protein Structure
with open(protein_hs_file, "r") as f:
    pdb_data = f.read()

# Open Ligand Structure
with open(ligand_pose_file, "r") as f:
    sdf_data = f.read()

# Initialize the viewer PDB file
viewer = py3Dmol.view(width=800, height=600)
viewer.addModel(pdb_data, "pdb")
viewer.addModel(sdf_data, "sdf")

# Style the Protein (Green); Cartoon Model
viewer.setStyle({'model': 0}, {'cartoon': {'color': 'green', 'opacity': 0.8}})

# Style the Ligand (Blue); Stick Model
viewer.setStyle({'model': 1}, {'stick': {'colorscheme': 'blueCarbon'}})

# Style Interacting Residues (Red)
viewer.addStyle({'model': 0, 'resi': res_ids}, 
                {'stick': {'color': 'red'}, 'cartoon': {'color': 'red'}})

# Add labels for the interacting residues
for res_name in interacting_res_list:
    res_num = int(''.join(filter(str.isdigit, res_name)))
    viewer.addLabel(res_name, 
                    {'fontSize': 10, 'fontColor': 'white', 'backgroundColor': 'black'},
                    {'model': 0, 'resi': res_num})

viewer.zoomTo()
viewer.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

END